In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

print("GPU configuration ready")

In [ ]:
!pip install -q \
    transformers==4.57.6 \
    datasets==4.8.5 \
    trl==0.29.1 \
    peft==0.21.2 \
    accelerate==1.15.0 \
    bitsandbytes==0.50.2 \
    huggingface_hub==0.36.2

In [ ]:
import torch
import transformers
import datasets
import trl
import peft
import accelerate
import bitsandbytes
import huggingface_hub

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print("All training libraries imported successfully!")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "PolyAI/banking77",
    revision="refs/pr/7"
)

labels = dataset["train"].features["label"].names

print("Training pool:", len(dataset["train"]))
print("Official test examples:", len(dataset["test"]))
print("Number of intents:", len(labels))

print("\nExample:")
print(dataset["train"][0])

In [ ]:
import random
import re

SEED = 42

def normalize(text):
    text = text.casefold()
    text = re.sub(r"[^\w ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Split only the original training partition.
# Never train or tune against the official test partition.
split = dataset["train"].train_test_split(
    test_size=0.1,
    stratify_by_column="label",
    seed=SEED
)

rng = random.Random(SEED)

# Keep validation separate from training.
validation_pool = list(split["test"])
train_pool = list(split["train"])

# Remove exact normalized duplicate texts across the
# held-out validation/test partitions and selected training data.
reserved_texts = {
    normalize(row["text"])
    for row in validation_pool
}

reserved_texts.update(
    normalize(row["text"])
    for row in dataset["test"]
)

def balanced_sample(rows, per_label, excluded):
    grouped = {i: [] for i in range(len(labels))}
    seen = set(excluded)

    shuffled = list(rows)
    rng.shuffle(shuffled)

    for row in shuffled:
        label_id = int(row["label"])
        key = normalize(row["text"])

        if key in seen:
            continue

        if len(grouped[label_id]) >= per_label:
            continue

        grouped[label_id].append(row)
        seen.add(key)

    selected = [
        row
        for label_id in range(len(labels))
        for row in grouped[label_id]
    ]

    rng.shuffle(selected)
    return selected

validation_raw = balanced_sample(
    validation_pool,
    per_label=5,
    excluded={
        normalize(row["text"])
        for row in dataset["test"]
    }
)

train_raw = balanced_sample(
    train_pool,
    per_label=25,
    excluded=reserved_texts
)

print("Training examples:", len(train_raw))
print("Validation examples:", len(validation_raw))
print("Official test held out:", len(dataset["test"]))

In [ ]:
from datasets import Dataset

print("Dataset imported successfully!")

In [ ]:
SYSTEM_PROMPT = (
    "You are a banking customer support intent classifier. "
    "Choose the single best banking intent label. "
    "Return only the intent label with no explanation."
)

def format_example(row):
    intent = labels[int(row["label"])]

    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": row["text"]
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": intent
            }
        ]
    }

train_sft = Dataset.from_list(
    [format_example(row) for row in train_raw]
)

validation_sft = Dataset.from_list(
    [format_example(row) for row in validation_raw]
)

print("Training:", len(train_sft))
print("Validation:", len(validation_sft))

print("\nSample:")
print(train_sft[0])

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

print("Model:", MODEL_ID)
print("Quantization: 4-bit NF4")
print("LoRA rank:", lora_config.r)

In [ ]:
training_args = SFTConfig(
    output_dir="/kaggle/working/qwen3-banking77-light",

    # Dataset processing
    max_length=192,
    completion_only_loss=True,

    # Lightweight training
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,

    # Learning rate
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch",

    # Tesla T4 precision
    fp16=True,
    bf16=False,

    # Previously successful checkpointing fix
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,

    # Save checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Pick checkpoint with lowest validation loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=20,
    report_to="none",
    seed=42,
    data_seed=42
)

print("Configuration ready")
print("Epochs:", training_args.num_train_epochs)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="eager"
)

print("Model loaded successfully")
print("Device:", model.device)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated(0) / 1024**3,
        2
    ),
    "GB"
)

In [ ]:
from collections import Counter

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    peft_config=lora_config
)

# Keep trainable parameters in FP32 to avoid
# the previous BF16 gradient-unscaling problem.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainer initialized")

print("Trainable parameter types:")
print(Counter(
    str(param.dtype)
    for param in trainer.model.parameters()
    if param.requires_grad
))

print("GPU count:", trainer.args.n_gpu)

In [ ]:
from collections import Counter

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    peft_config=lora_config
)

# Keep trainable parameters in FP32 to avoid
# the previous BF16 gradient-unscaling problem.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainer initialized")

print("Trainable parameter types:")
print(Counter(
    str(param.dtype)
    for param in trainer.model.parameters()
    if param.requires_grad
))

print("GPU count:", trainer.args.n_gpu)

In [ ]:
result = trainer.train()

print("=" * 45)
print("TRAINING COMPLETED")
print("=" * 45)

print("Training loss:", result.training_loss)
print("Completed epochs:", trainer.state.epoch)
print("Completed steps:", trainer.state.global_step)
print("Best validation loss:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
adapter_dir = "/kaggle/working/qwen3-banking77-light-final"

trainer.save_model(adapter_dir)
trainer.processing_class.save_pretrained(adapter_dir)

print("Model adapter saved:", adapter_dir)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

model = trainer.model
model.eval()

tokenizer = trainer.processing_class
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model ready for evaluation")
print("Validation examples:", len(validation_raw))
print("Number of classes:", len(labels))

In [ ]:
SYSTEM_PROMPT = (
    "You are a banking customer support intent classifier. "
    "Choose the single best banking intent label. "
    "Return only the intent label with no explanation."
)

@torch.inference_mode()
def predict_intent(text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": text
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt"
    ).to("cuda:0")

    outputs = model.generate(
        inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0, inputs.shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prediction

In [ ]:
examples = [
    "I lost my bank card yesterday",
    "Why was I charged twice?",
    "I forgot my PIN",
    "My card hasn't arrived yet",
    "How do I exchange euros for pounds?"
]

for message in examples:
    prediction = predict_intent(message)

    print("Customer:", message)
    print("Predicted intent:", prediction)
    print("-" * 50)

In [ ]:
# The same validation subset used during training.
# Do not include official BANKING77 test examples here.

label_set = set(labels)

def normalize_prediction(text):
    return text.strip().strip("`'\"").strip()

records = []

for row in tqdm(validation_raw, desc="Evaluating"):
    customer_text = row["text"]
    true_label = labels[int(row["label"])]

    raw_prediction = predict_intent(customer_text)
    predicted_label = normalize_prediction(raw_prediction)

    # Invalid outputs are counted as incorrect.
    is_valid = predicted_label in label_set

    records.append({
        "text": customer_text,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "valid_label": is_valid,
        "correct": predicted_label == true_label
    })

results_df = pd.DataFrame(records)

print("Evaluated:", len(results_df))
print("Correct predictions:", results_df["correct"].sum())

In [ ]:
y_true = results_df["true_label"].tolist()
y_pred = results_df["predicted_label"].tolist()

accuracy = accuracy_score(y_true, y_pred)

# Explicitly count predictions outside the 77 valid labels
# as classification errors.
macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=labels,
    average="macro",
    zero_division=0
)

invalid_count = (~results_df["valid_label"]).sum()

print("=" * 50)
print("QWEN3 BANKING77 VALIDATION RESULTS")
print("=" * 50)

print(f"Accuracy: {accuracy:.2%}")
print(f"Macro-F1: {macro_f1:.4f}")
print(f"Invalid predictions: {invalid_count}")
print(f"Total evaluated: {len(results_df)}")

In [ ]:
errors = results_df[
    results_df["correct"] == False
]

print("Total incorrect:", len(errors))

display(
    errors[
        ["text", "true_label", "predicted_label"]
    ].head(20)
)


In [ ]:
import json
from pathlib import Path

results_df.to_csv(
    "/kaggle/working/validation_predictions.csv",
    index=False
)

metrics = {
    "model": "Qwen/Qwen3-1.7B",
    "experiment": "BANKING77 lightweight supervised QLoRA",
    "evaluation_split": "validation",
    "evaluation_examples": len(results_df),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "invalid_predictions": int(invalid_count),
    "training_epochs": 1,
    "lora_rank": 4,
    "inference_thinking": False
}

Path("/kaggle/working/validation_metrics.json").write_text(
    json.dumps(metrics, indent=2)
)

print(json.dumps(metrics, indent=2))

In [ ]:
invalid_df = results_df[
    results_df["valid_label"] == False
]

print("Total invalid predictions:", len(invalid_df))

display(
    invalid_df[
        ["text", "true_label", "predicted_label"]
    ].head(30)
)

In [ ]:
print("Invalid prediction examples:")

for pred in invalid_df["predicted_label"].head(20):
    print(repr(pred))

In [ ]:
@torch.inference_mode()
def predict_intent_v2(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt"
    ).to("cuda:0")

    outputs = model.generate(
        inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    generated = outputs[0, inputs.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

print("Comparing old and new predictions:\n")

for _, row in invalid_df.head(10).iterrows():
    new_pred = predict_intent_v2(row["text"])

    print("Customer:", row["text"])
    print("Expected:", row["true_label"])
    print("Old:", repr(row["predicted_label"]))
    print("New:", repr(new_pred))
    print("Valid:", new_pred in set(labels))
    print("-" * 60)

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

model = trainer.model
model.eval()
tokenizer = trainer.processing_class

official_labels = list(labels)

@torch.inference_mode()
def predict_closed_set(text, batch_size=8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text}
    ]

    # Match the inference prompt used previously.
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False
    )

    scores = []

    # Process 8 candidate labels at a time to limit VRAM.
    for start in range(0, len(official_labels), batch_size):
        candidates = official_labels[start:start + batch_size]

        candidate_ids = [
            tokenizer.encode(label, add_special_tokens=False)
            for label in candidates
        ]

        sequences = [
            prompt_ids + ids
            for ids in candidate_ids
        ]

        max_len = max(len(seq) for seq in sequences)

        pad_id = tokenizer.pad_token_id

        input_ids = torch.tensor([
            seq + [pad_id] * (max_len - len(seq))
            for seq in sequences
        ], device="cuda:0")

        attention_mask = torch.tensor([
            [1] * len(seq) + [0] * (max_len - len(seq))
            for seq in sequences
        ], device="cuda:0")

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        log_probs = F.log_softmax(
            outputs.logits.float(), dim=-1
        )

        prompt_len = len(prompt_ids)

        for i, ids in enumerate(candidate_ids):
            # Causal LM predicts token at position j
            # using logits from position j-1.
            positions = torch.arange(
                prompt_len - 1,
                prompt_len + len(ids) - 1,
                device="cuda:0"
            )

            tokens = torch.tensor(ids, device="cuda:0")

            token_scores = log_probs[i, positions, tokens]

            # Average log-likelihood reduces the
            # preference for shorter labels.
            score = token_scores.mean().item()
            scores.append(score)

        del outputs, log_probs

    best_idx = max(range(len(scores)), key=lambda i: scores[i])

    return official_labels[best_idx], scores[best_idx]

In [ ]:
test_rows = invalid_df.head(10)

comparison = []

for _, row in tqdm(test_rows.iterrows(), total=len(test_rows)):
    new_prediction, score = predict_closed_set(row["text"])

    comparison.append({
        "text": row["text"],
        "expected": row["true_label"],
        "old_prediction": row["predicted_label"],
        "new_prediction": new_prediction,
        "new_correct": new_prediction == row["true_label"]
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

print(
    "Correct with closed-set scoring:",
    comparison_df["new_correct"].sum(),
    "out of",
    len(comparison_df)
)

In [ ]:
import shutil

adapter_dir = "/kaggle/working/qwen3-banking77-light-final"

zip_path = shutil.make_archive(
    "/kaggle/working/qwen3-banking77-light-final",
    "zip",
    adapter_dir
)

print("Created:", zip_path)

In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/qwen3-banking77-light-final.zip"
))